In [4]:
import os
import re
import html
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from sacrebleu.metrics import CHRF # Standard for character-level overlap

# --- CONFIGURATION ---
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
CACHE_DIR = Path("./triple_eval_cache")
CACHE_DIR.mkdir(exist_ok=True)

TRIPLE_DATA_CACHE = CACHE_DIR / "parsed_triples.csv"
TRIPLE_RESULTS_CACHE = CACHE_DIR / "triple_evaluation_results.csv"

# --- UTILS ---
def extract_digits(text):
    return set(re.findall(r"\d+(?:[.,]\d+)?", str(text)))

def split_triple(t_str):
    """Splits 'Subj | Pred | Obj' into components."""
    parts = [p.strip() for p in str(t_str).split('|')]
    if len(parts) == 3:
        return parts[0], parts[1], parts[2]
    return "", "", ""

def parse_webnlg_triples(root_path):
    if TRIPLE_DATA_CACHE.exists():
        print("Loading triples from cache...")
        return pd.read_csv(TRIPLE_DATA_CACHE, converters={'ref_triples': eval, 'tgt_triples': eval})

    data = []
    root = Path(root_path)
    print("Parsing XML for triplesets...")
    
    for xml_file in tqdm(list(root.rglob("*.xml"))):
        tree = ET.parse(xml_file)
        for entry in tree.findall(".//entry"):
            eid = entry.get("eid")
            ref_triples = [t.text for t in entry.findall(".//modifiedtripleset/mtriple")]
            
            targets = {
                'es': [t.text for t in entry.findall(".//spanishtripleset/striple")],
                'ca': [t.text for t in entry.findall(".//catalantripleset/ctriple")],
                'en_bt': [t.text for t in entry.findall(".//enbttripleset/bttriple")]
            }
            
            for lang, tgt_triples in targets.items():
                if ref_triples and tgt_triples:
                    # Join components for semantic evaluation
                    ref_str = " | ".join(ref_triples)
                    tgt_str = " | ".join(tgt_triples)
                    
                    # Extract predicates and entities separately
                    ref_preds = [split_triple(t)[1] for t in ref_triples]
                    tgt_preds = [split_triple(t)[1] for t in tgt_triples]
                    ref_ents = [f"{split_triple(t)[0]} {split_triple(t)[2]}" for t in ref_triples]
                    tgt_ents = [f"{split_triple(t)[0]} {split_triple(t)[2]}" for t in tgt_triples]

                    data.append({
                        'eid': eid, 'lang': lang,
                        'ref_triples': ref_triples, 'tgt_triples': tgt_triples,
                        'ref_str': ref_str, 'tgt_str': tgt_str,
                        'ref_preds': " ".join(ref_preds), 'tgt_preds': " ".join(tgt_preds),
                        'ref_ents': " ".join(ref_ents), 'tgt_ents': " ".join(tgt_ents)
                    })
                    
    df = pd.DataFrame(data)
    df.to_csv(TRIPLE_DATA_CACHE, index=False)
    return df

# --- EVALUATOR ---
class TripleSetEvaluator:
    def __init__(self, model_name="intfloat/multilingual-e5-base", batch_size=32):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=self.device)
        self.batch_size = batch_size
        self.chrf = CHRF()

    def get_embeddings(self, texts):
        """Standard embeddings (no prefix) for identity/alignment matching."""
        return self.model.encode(texts, convert_to_tensor=True, 
                                 batch_size=self.batch_size, show_progress_bar=False)

    def evaluate(self, df):
        # 1. Structural, Literal, and Lexical Metrics (RQ1)
        print("Calculating Structural, Literal, and Lexical metrics...")
        results = []
        for _, row in tqdm(df.iterrows(), total=len(df)):
            ref, tgt = row['ref_triples'], row['tgt_triples']
            
            # Structural Parity
            parity = 1.0 if len(ref) == len(tgt) else 0.0
            
            # Digit Preservation
            ref_digits = extract_digits(" ".join(ref))
            tgt_digits = extract_digits(" ".join(tgt))
            digit_score = 1.0 if not ref_digits else len(ref_digits & tgt_digits) / len(ref_digits)
            
            # Lexical chrF (Character-level overlap)
            chrf_score = self.chrf.sentence_score(row['tgt_str'], [row['ref_str']]).score / 100
            
            results.append({
                'structural_parity': parity,
                'digit_preservation': digit_score,
                'chrf_lexical_alignment': chrf_score
            })
        
        df = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)

        # 2. Semantic Alignment (RQ2) - Segmented by Components
        tasks = {
            'holistic': ('ref_str', 'tgt_str'),
            'predicates': ('ref_preds', 'tgt_preds'),
            'entities': ('ref_ents', 'tgt_ents')
        }

        for name, (ref_col, tgt_col) in tasks.items():
            print(f"Computing Semantic Alignment for {name}...")
            ref_emb = self.get_embeddings(df[ref_col].tolist())
            tgt_emb = self.get_embeddings(df[tgt_col].tolist())
            
            ref_emb = torch.nn.functional.normalize(ref_emb, p=2, dim=1)
            tgt_emb = torch.nn.functional.normalize(tgt_emb, p=2, dim=1)
            sims = (ref_emb * tgt_emb).sum(dim=1).cpu().tolist()
            
            df[f'semantic_{name}_alignment'] = sims
            
        df.to_csv(TRIPLE_RESULTS_CACHE, index=False)
        return df

# --- REPORTING ---
def analyze_triple_results(df):
    metrics = [
        'structural_parity', 
        'digit_preservation', 
        'chrf_lexical_alignment',
        'semantic_holistic_alignment', 
        'semantic_predicates_alignment', 
        'semantic_entities_alignment'
    ]
    report = []
    for lang in df['lang'].unique():
        ldf = df[df['lang'] == lang]
        row = {'Lang': lang, 'Count': len(ldf)}
        for m in metrics:
            row[f'{m}_Mean'] = round(ldf[m].mean(), 4)
        report.append(row)
    return pd.DataFrame(report)

# --- EXECUTION ---
df_triples = parse_webnlg_triples("WebNLG_CA_BT")
evaluator = TripleSetEvaluator()
df_results = evaluator.evaluate(df_triples)
summary = analyze_triple_results(df_results)

print("\n--- EXTENDED TRIPLESET EVALUATION SUMMARY ---")
print(summary.set_index('Lang').T)

Parsing XML for triplesets...


  0%|          | 0/187 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calculating Structural, Literal, and Lexical metrics...


  0%|          | 0/56844 [00:00<?, ?it/s]

Computing Semantic Alignment for holistic...
Computing Semantic Alignment for predicates...
Computing Semantic Alignment for entities...

--- EXTENDED TRIPLESET EVALUATION SUMMARY ---
Lang                                        es          ca       en_bt
Count                               18948.0000  18948.0000  18948.0000
structural_parity_Mean                  1.0000      1.0000      1.0000
digit_preservation_Mean                 0.9783      0.9860      1.0000
chrf_lexical_alignment_Mean             0.5097      0.5298      0.9978
semantic_holistic_alignment_Mean        0.9479      0.9433      0.9998
semantic_predicates_alignment_Mean      0.9056      0.9000      1.0000
semantic_entities_alignment_Mean        0.9579      0.9527      0.9997


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_knee_point(values):
    """Geometric knee point detection."""
    if len(values) < 3:
        return np.mean(values) if len(values) > 0 else 0
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    coords = np.column_stack((np.linspace(0, 1, n), sorted_vals))
    line = coords[-1] - coords[0]
    line_norm = line / np.linalg.norm(line)
    vec_start = coords - coords[0]
    dist = np.linalg.norm(vec_start - np.outer(np.dot(vec_start, line_norm), line_norm), axis=1)
    return sorted_vals[np.argmax(dist)]

def format_with_gap(value, baseline=1.0):
    """Formats a value with its gap from the baseline: 0.978 (-0.022)."""
    gap = value - baseline
    # Use + or - sign automatically for the gap
    return f"{value:.2f} ({gap:+.2f})"

def create_integrated_gap_table(csv_path, output_xlsx):
    # Load the triple evaluation results
    df = pd.read_csv(csv_path)
    
    metrics = [
        'structural_parity', 
        'digit_preservation', 
        'chrf_lexical_alignment',
        'semantic_holistic_alignment', 
        'semantic_predicates_alignment', 
        'semantic_entities_alignment'
    ]
    
    GOLD_BASELINE = 1.0
    final_rows = []

    # Iterate through all languages (ES, CA, and now EN_BT)
    for lang in df['lang'].unique():
        ldf = df[df['lang'] == lang].copy()
        
        # Knee Logic based on Holistic Semantic Alignment
        thresh = find_knee_point(ldf['semantic_holistic_alignment'].values)
        under = ldf[ldf['semantic_holistic_alignment'] < thresh]
        over = ldf[ldf['semantic_holistic_alignment'] >= thresh]
        tail_pct = len(under) / len(ldf) * 100

        # Segments to display
        segments = {
            'Full Mean': ldf,
            'Under-Knee': under,
            'Over-Knee': over
        }

        for seg_name, seg_df in segments.items():
            # Initial column data
            row_data = {
                'Language': lang.upper(),
                'Segment': seg_name,
                'Knee Threshold': f"{thresh:.3f}",
                'Tail %': f"{tail_pct:.2f}%"
            }
            
            # Compute formatted metrics with integrated gaps
            for m in metrics:
                col_name = m.replace('_', ' ').replace('chrf', 'chrF').title()
                mean_val = seg_df[m].mean()
                
                # Format: 0.9078 (-0.0922)
                row_data[col_name] = format_with_gap(mean_val, GOLD_BASELINE)
            
            final_rows.append(row_data)

    # Build the wide DataFrame
    summary_df = pd.DataFrame(final_rows)
    summary_df.set_index(['Language', 'Segment'], inplace=True)

    # Export to Excel
    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='Triple_Integrated_Gaps')
        
        # Adjust column widths for better readability in paper drafts
        worksheet = writer.sheets['Triple_Integrated_Gaps']
        for column_cells in worksheet.columns:
            # We add extra width for the (±0.0000) part
            length = max(len(str(cell.value) or "") for cell in column_cells)
            worksheet.column_dimensions[column_cells[0].column_letter].width = length + 2

    print(f"Integrated Gap Table saved to: {output_xlsx}")

if __name__ == "__main__":
    INPUT_CSV = "triple_eval_cache/triple_evaluation_results.csv"
    OUTPUT_FILE = "WebNLG_Triple_Integrated_Summary.xlsx"
    
    if Path(INPUT_CSV).exists():
        create_integrated_gap_table(INPUT_CSV, OUTPUT_FILE)
    else:
        print("Error: Input CSV not found.")

Integrated Gap Table saved to: WebNLG_Triple_Integrated_Summary.xlsx
